In [1]:
DATASET = "wine"
SPLIT = 1
SEED = 0

NUM_INDUCING = 200
LENGTHSCALE_MIN = 1.0e-3
LENGTHSCALE_MAX = 1.0e3

NUM_PARTICLES = 32
STEP_SIZE = 1.0e-4

SIGMA_MIN = 0.1
SIGMA_MAX = 100.0
SIGMA_LR = 0.1
KERNEL_LR = 0.01

ALGORITHM = "sgld"
SGLD_BATCH_SIZE = 512

C_GRID = [10, 25, 50, 100]
VAL_FRACTION = 0.3
ADAPT_TARGET = None

NUM_ADAPT_STEPS = 15000
WARMUP_STEPS = 10000
SIGMA_ADAPT_STEPS = 500
KERNEL_ADAPT_STEPS = 0

NUM_SAMPLE_STEPS = 3000
BURN_FRACTION = 0.5
THIN = 50

In [2]:
import math

import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np

import gpjax as gpx
import optax as ox
import paramax as px

from sklearn.preprocessing import StandardScaler

from non_parametric_pro import ula
from non_parametric_pro.ula import parametric_ula
from non_parametric_pro.sgld import parametric_sgld, sgld
from non_parametric_pro.density import ProParameters, pro_logdensity_fn
from non_parametric_pro.util import (
    nlpd_gp,
    nlpd_pro,
    prediction_basis,
    run_inference_algorithm_with_burn_in,
    train_val_split,
)
from non_parametric_pro.inducing import PointInducingBasis, compute_inducing_basis, kmeans_inducing_points
from non_parametric_pro.parameter_adaptation import parameter_adaptation
from non_parametric_pro.data.uci.uci import load_uci_regression_dataset

jax.config.update("jax_enable_x64", True)


def adaptation_algorithm():
    return ula if ALGORITHM == "ula" else sgld(batch_size=SGLD_BATCH_SIZE)

def sampling_algorithm(pro_params):
    if ALGORITHM == "ula":
        return parametric_ula(pro_logdensity_fn, pro_params)
    return parametric_sgld(pro_logdensity_fn, pro_params, batch_size=SGLD_BATCH_SIZE)

In [3]:
key = jr.PRNGKey(SEED)

example = load_uci_regression_dataset(DATASET, split=SPLIT)

scaler_x = StandardScaler()
scaler_y = StandardScaler()
x_train = scaler_x.fit_transform(example.x_train)
y_train = scaler_y.fit_transform(example.y_train)
x_test = scaler_x.transform(example.x_test)
y_test = scaler_y.transform(example.y_test)

n = x_train.shape[0]
print("N_train:", n, " N_test:", x_test.shape[0], " D:", x_train.shape[1])

N_train: 1280  N_test: 319  D: 11


In [4]:
D = x_train.shape[1]

data = gpx.Dataset(X=x_train, y=y_train)
lengthscale = gpx.parameters.SigmoidBounded(
    jnp.sqrt(D) * jnp.ones((D,)), low=LENGTHSCALE_MIN, high=LENGTHSCALE_MAX
)
kernel = gpx.kernels.RBF(lengthscale=lengthscale, variance=px.NonTrainable(jnp.array(1.0)))
prior = gpx.gps.Prior(mean_function=gpx.mean_functions.Zero(), kernel=kernel)
likelihood = gpx.likelihoods.Gaussian(num_datapoints=data.n, obs_stddev=jnp.sqrt(0.01))
posterior = prior * likelihood

key, km_key = jr.split(key)
z_init = kmeans_inducing_points(km_key, jnp.array(x_train), NUM_INDUCING).z

variational_family = gpx.variational_families.CollapsedVariationalGaussian(
    posterior=posterior,
    inducing_inputs=z_init,
)
opt_vf, _ = gpx.fit_scipy(
    model=variational_family,
    objective=lambda p, d: -gpx.objectives.collapsed_elbo(p, d),
    train_data=data,
    verbose=False,
)

vgp_kernel = opt_vf.posterior.prior.kernel
vgp_sigma = opt_vf.posterior.likelihood.obs_stddev
z_opt = px.unwrap(opt_vf.inducing_inputs)

vgp_latent_train_dist = opt_vf.predict(x_train, train_data=data)
vgp_predictive_train_dist = opt_vf.posterior.likelihood(vgp_latent_train_dist)
vgp_predictive_train_mean = vgp_predictive_train_dist.mean
vgp_predictive_train_std = jnp.sqrt(vgp_predictive_train_dist.variance)

vgp_latent_dist = opt_vf.predict(x_test, train_data=data)
vgp_predictive_dist = opt_vf.posterior.likelihood(vgp_latent_dist)
vgp_predictive_mean = vgp_predictive_dist.mean
vgp_predictive_std = jnp.sqrt(vgp_predictive_dist.variance)

print("VGP sigma:", float(px.unwrap(vgp_sigma)))
print("VGP Train NLPD:", nlpd_gp(y_train, vgp_predictive_train_mean, vgp_predictive_train_std))
print("VGP Test NLPD:", nlpd_gp(y_test, vgp_predictive_mean, vgp_predictive_std))

VGP sigma: 0.40007366315880943
VGP Train NLPD: 0.4494528963484493
VGP Test NLPD: 0.5726326149432434


In [5]:
kernel = vgp_kernel
sigma_init_val = float(px.unwrap(vgp_sigma))

inducing_basis = PointInducingBasis(z=z_opt)
basis_full, residual_std_full = compute_inducing_basis(inducing_basis, kernel, x_train)
basis_dim = basis_full.shape[1]

c_val_nlpd = []
c_results = []
for c in C_GRID:
    alpha = float(c) / math.sqrt(n)
    key, split_key, pos_key, adapt_key = jr.split(key, 4)
    split = train_val_split(split_key, x_train, y_train, val_fraction=VAL_FRACTION)

    fold_params = ProParameters(
        y=split.y_train, basis=None, step_size=STEP_SIZE,
        sigma=gpx.parameters.SigmoidBounded(sigma_init_val, low=SIGMA_MIN, high=SIGMA_MAX),
        alpha=alpha, residual_std=None,
    )
    initial_position = jr.normal(pos_key, (basis_dim, NUM_PARTICLES))

    adaptation = parameter_adaptation(
        adaptation_algorithm(),
        pro_logdensity_fn,
        fold_params,
        x_train=split.x_train,
        initial_kernel=kernel,
        warmup_steps=WARMUP_STEPS,
        sigma_adapt_steps=SIGMA_ADAPT_STEPS,
        kernel_adapt_steps=KERNEL_ADAPT_STEPS,
        objective_fn=pro_logdensity_fn,
        inducing_basis=inducing_basis,
        x_val=split.x_val,
        y_val=split.y_val,
        adapt_target=ADAPT_TARGET,
        sigma_optimizer=ox.adam(SIGMA_LR),
        kernel_optimizer=ox.adam(KERNEL_LR),
        progress_bar=True,
    )
    adaptation_results, adaptation_info = adaptation.run(
        adapt_key, initial_position, num_steps=NUM_ADAPT_STEPS
    )
    adapted_kernel_c = (
        px.unwrap(jax.tree.map(lambda x: x[-1], adaptation_info.kernel))
        if KERNEL_ADAPT_STEPS > 0 else kernel
    )
    val_basis, val_cov = prediction_basis(
        adapted_kernel_c, split.x_train, split.x_val,
        adaptation_results.parameters, inducing_basis=inducing_basis,
    )
    val_nlpd = float(nlpd_pro(
        split.y_val, val_basis, val_cov, adaptation_results.state.position,
        parameters=adaptation_results.parameters,
    ))
    print(
        f"c={c:.4g}  alpha={alpha:.4g}  "
        f"adapted sigma={float(np.array(px.unwrap(adaptation_results.parameters.sigma))):.4f}  "
        f"val NLPD={val_nlpd:.4f}"
    )
    c_val_nlpd.append(val_nlpd)
    c_results.append((adaptation_results, adapted_kernel_c))

Running parameter adaptation


<div><progress max="15000" value="15000"></progress> 100.00% [15000/15000 00:00&lt;?]</div>

c=10  alpha=0.2795  adapted sigma=0.3211  val NLPD=0.7003
Running parameter adaptation


<div><progress max="15000" value="15000"></progress> 100.00% [15000/15000 00:00&lt;?]</div>

c=25  alpha=0.6988  adapted sigma=0.2477  val NLPD=0.5685
Running parameter adaptation


<div><progress max="15000" value="15000"></progress> 100.00% [15000/15000 00:00&lt;?]</div>

c=50  alpha=1.398  adapted sigma=0.3728  val NLPD=0.6889
Running parameter adaptation


<div><progress max="15000" value="15000"></progress> 100.00% [15000/15000 00:00&lt;?]</div>

c=100  alpha=2.795  adapted sigma=0.2859  val NLPD=0.5441


In [6]:
best_idx = int(np.argmin(c_val_nlpd))
best_c = float(C_GRID[best_idx])
best_alpha = best_c / math.sqrt(n)
adaptation_results, adapted_kernel = c_results[best_idx]
adapted_sigma = float(np.array(px.unwrap(adaptation_results.parameters.sigma)))
print(f"Best c={best_c:.4g} (alpha={best_alpha:.4g}, val NLPD={c_val_nlpd[best_idx]:.4f})")

if KERNEL_ADAPT_STEPS > 0:
    basis_full, residual_std_full = compute_inducing_basis(inducing_basis, adapted_kernel, x_train)

pro_params = ProParameters(
    y=y_train,
    basis=basis_full,
    step_size=STEP_SIZE,
    sigma=adaptation_results.parameters.sigma,
    alpha=best_alpha,
    residual_std=residual_std_full,
)
algorithm = sampling_algorithm(pro_params)
key, sample_key = jr.split(key)
_, (states, _) = run_inference_algorithm_with_burn_in(
    rng_key=sample_key,
    inference_algorithm=algorithm,
    num_steps=NUM_SAMPLE_STEPS,
    burn_ratio=BURN_FRACTION,
    initial_position=adaptation_results.state.position,
    progress_bar=True,
)
particles = states.position[::THIN]

Best c=100 (alpha=2.795, val NLPD=0.5441)


<div><progress max="1500" value="1500"></progress> 100.00% [1500/1500 00:00&lt;?]</div>

<div><progress max="1500" value="1500"></progress> 100.00% [1500/1500 00:00&lt;?]</div>

In [7]:
test_basis, test_cov = prediction_basis(
    adapted_kernel, x_train, x_train, pro_params, inducing_basis=inducing_basis
)
print("PRO Train NLPD:", nlpd_pro(y_train, test_basis, test_cov, particles, parameters=pro_params))

test_basis, test_cov = prediction_basis(
    adapted_kernel, x_train, x_test, pro_params, inducing_basis=inducing_basis
)
print("PRO Test NLPD:", nlpd_pro(y_test, test_basis, test_cov, particles, parameters=pro_params))
print("VGP Train NLPD:", nlpd_gp(y_train, vgp_predictive_train_mean, vgp_predictive_train_std))
print("VGP Test NLPD:", nlpd_gp(y_test, vgp_predictive_mean, vgp_predictive_std))

PRO Train NLPD: 0.26330229953666706
PRO Test NLPD: 0.49120308336060603
VGP Train NLPD: 0.4494528963484493
VGP Test NLPD: 0.5726326149432434
